In [1]:
import torch
import torch.nn as nn

In [2]:
text = "hello world"

unique_chars = set()
for char in text:
    unique_chars.add(char)

chars = sorted(unique_chars)
vocab_size = len(chars)

stoi = {}
itos = {}

for i, char in enumerate(chars):
    stoi[char] = i
    itos[i] = char

print("字符 → ID:", stoi)
print("ID → 字符:", itos)

encoded = []

for char in text:
    encoded.append(stoi[char])

x = torch.tensor(encoded[:-1], dtype= torch.long)
y = torch.tensor(encoded[1:], dtype= torch.long)
table = nn.Embedding(num_embeddings=vocab_size,embedding_dim=vocab_size)
loss_api = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(table.parameters(),lr= 0.1)

with torch.no_grad():
    initial_loss = loss_api(table(x), y)
    print(f"Initial loss: {initial_loss.item():.6f}")


for step in range(1000):
    optimizer.zero_grad(set_to_none=True)
    pred = table(x)
    loss = loss_api(pred,y)
    loss.backward()
    optimizer.step()

    if (step + 1) % 200 == 0:
        print(f"Step {step + 1}: loss={loss.item():.6f}")



with torch.no_grad():
    final_loss = loss_api(table(x), y)
    print(f"Final loss: {final_loss.item():.6f}")
        






字符 → ID: {' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}
ID → 字符: {0: ' ', 1: 'd', 2: 'e', 3: 'h', 4: 'l', 5: 'o', 6: 'r', 7: 'w'}
Initial loss: 2.630423
Step 200: loss=1.223321
Step 400: loss=0.801988
Step 600: loss=0.662790
Step 800: loss=0.602265
Step 1000: loss=0.569616
Final loss: 0.569491


In [3]:
with torch.no_grad():
    l_logits = table.weight[stoi["l"]]
    probabilities = torch.softmax(l_logits, dim=0)

    for char, probability in zip(chars, probabilities):
        print(f"{repr(char)}: {probability.item():.4f}")

    print("概率之和:", probabilities.sum().item())
    next_id = torch.multinomial(probabilities, num_samples=1)
    print(next_id)
    print("抽到的字符:", repr(itos[next_id.item()]))

' ': 0.0112
'd': 0.3119
'e': 0.0123
'h': 0.0099
'l': 0.3119
'o': 0.3119
'r': 0.0110
'w': 0.0199
概率之和: 1.0
tensor([5])
抽到的字符: 'o'


In [5]:
output = ["h"]
with torch.no_grad():
    x = "h"
    for step in range(20):
        
        x_logit = table.weight[stoi[x]]
        prob = torch.softmax(x_logit,dim=0)
        nxt_id = torch.multinomial(prob,num_samples=1)
        output.append(str(itos[nxt_id.item()]))
        x = str(itos[nxt_id.item()])

    print(output)
    print(len(output))

['h', 'e', 'l', 'l', 'l', 'w', 'o', 'r', 'l', 'l', 'l', 'o', ' ', 'w', 'o', ' ', 'w', 'o', 'r', 'w', 'w']
21


In [6]:
print(repr("".join(output)))

'helllworlllo wo worww'
